# Notebook 04 — Curriculum and Topic Mapping

## Principle
Mapping confidence is measured against a manual gold set.
We do not claim high confidence until agreement is quantified.

## This notebook phase
1. Paths and inputs
2. Taxonomy v1
3. Load question structure
4. Create 2025 P1 gold-set template
5. Save gold set for manual labelling

Mapping rules come after the gold set exists.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import re
import json

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
META_DIR = DATA_DIR / "metadata"
PROCESSED_DIR = DATA_DIR / "processed"
QUESTIONS_DIR = PROCESSED_DIR / "questions"
TAXONOMY_DIR = PROCESSED_DIR / "taxonomy"
GOLD_DIR = PROCESSED_DIR / "gold_sets"

for d in [QUESTIONS_DIR, TAXONOMY_DIR, GOLD_DIR, META_DIR]:
    d.mkdir(parents=True, exist_ok=True)

question_structure_path = QUESTIONS_DIR / "question_structure.csv"
taxonomy_path = TAXONOMY_DIR / "taxonomy_math_v1.csv"
gold_template_path = GOLD_DIR / "gold_2025_p1_template.csv"
gold_labelled_path = GOLD_DIR / "gold_2025_p1_labelled.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Questions:", question_structure_path.exists())

PROJECT_ROOT: c:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence
Questions: True


In [2]:
taxonomy_rows = [
    # topic, subtopic, skill, caps_section, notes
    ("Algebra & Equations", "Linear equations", "Solve linear equations", "Algebra", "v1"),
    ("Algebra & Equations", "Quadratic equations", "Solve quadratics by factorisation", "Algebra", "v1"),
    ("Algebra & Equations", "Quadratic equations", "Solve quadratics by formula", "Algebra", "v1"),
    ("Algebra & Equations", "Inequalities", "Solve and represent inequalities", "Algebra", "v1"),
    ("Algebra & Equations", "Algebraic manipulation", "Simplify/factorise/expand expressions", "Algebra", "v1"),
    ("Algebra & Equations", "Exponents & surds", "Simplify exponential and surd expressions", "Algebra", "v1"),

    ("Number Patterns & Sequences", "Arithmetic sequences", "Find nth term / sum of arithmetic sequence", "Patterns", "v1"),
    ("Number Patterns & Sequences", "Geometric sequences", "Find nth term / sum of geometric sequence", "Patterns", "v1"),
    ("Number Patterns & Sequences", "Quadratic patterns", "Determine quadratic pattern rules", "Patterns", "v1"),

    ("Functions & Graphs", "Linear functions", "Interpret or sketch linear functions", "Functions", "v1"),
    ("Functions & Graphs", "Quadratic functions", "Interpret/sketch parabola and parameters", "Functions", "v1"),
    ("Functions & Graphs", "Hyperbola", "Interpret/sketch hyperbola", "Functions", "v1"),
    ("Functions & Graphs", "Exponential functions", "Interpret/sketch exponential functions", "Functions", "v1"),
    ("Functions & Graphs", "Inverse functions", "Determine or interpret inverses", "Functions", "v1"),

    ("Finance", "Compound interest", "Apply compound growth formulae", "Finance", "v1"),
    ("Finance", "Decay / reduction", "Apply reduction formulae", "Finance", "v1"),
    ("Finance", "Annuities", "Present/future value calculations", "Finance", "v1"),

    ("Calculus", "First principles", "Differentiate from first principles", "Calculus", "v1"),
    ("Calculus", "Rules of differentiation", "Apply differentiation rules", "Calculus", "v1"),
    ("Calculus", "Cubic graphs", "Analyse cubic using calculus", "Calculus", "v1"),
    ("Calculus", "Optimisation", "Solve optimisation problems", "Calculus", "v1"),

    ("Probability", "Basic probability", "Compute simple probabilities", "Probability", "v1"),
    ("Probability", "Venn / mutually exclusive", "Use set relationships in probability", "Probability", "v1"),
    ("Probability", "Tree diagrams / counting", "Use tree diagrams or counting principles", "Probability", "v1"),

    ("Trigonometry", "Identities", "Prove or apply trig identities", "Trigonometry", "v1"),
    ("Trigonometry", "Equations", "Solve trigonometric equations", "Trigonometry", "v1"),
    ("Trigonometry", "2D/3D applications", "Solve trig application problems", "Trigonometry", "v1"),
    ("Trigonometry", "Graphs", "Interpret/sketch trig graphs", "Trigonometry", "v1"),

    ("Euclidean Geometry", "Circle geometry", "Apply circle theorems", "Geometry", "v1"),
    ("Euclidean Geometry", "Similarity / proportion", "Use similarity and proportion", "Geometry", "v1"),
    ("Euclidean Geometry", "Riders / proofs", "Complete geometric riders/proofs", "Geometry", "v1"),

    ("Analytical Geometry", "Distance / midpoint / gradient", "Apply analytic geometry formulae", "Analytical Geometry", "v1"),
    ("Analytical Geometry", "Equation of a line", "Determine equation of a line", "Analytical Geometry", "v1"),
    ("Analytical Geometry", "Circles", "Equation/properties of a circle", "Analytical Geometry", "v1"),

    ("Statistics", "Regression / correlation", "Interpret regression and correlation", "Statistics", "v1"),
    ("Statistics", "Representations", "Interpret statistical representations", "Statistics", "v1"),

    ("Mixed / Multi-topic", "Mixed", "Multi-topic assessed skill", "Mixed", "v1"),
    ("Unmapped", "Unmapped", "Insufficient evidence to map", "Unmapped", "v1"),
]

taxonomy_df = pd.DataFrame(
    taxonomy_rows,
    columns=["topic", "subtopic", "skill", "caps_section", "taxonomy_version"]
)

taxonomy_df.to_csv(taxonomy_path, index=False)
print("Taxonomy rows:", len(taxonomy_df))
print("Topics:", taxonomy_df["topic"].nunique())
display(taxonomy_df.head(12))

Taxonomy rows: 38
Topics: 12


,topic,subtopic,skill,caps_section,taxonomy_version
0,Algebra & Equations,Linear equations,Solve linear equations,Algebra,v1
1,Algebra & Equations,Quadratic equations,Solve quadratics by factorisation,Algebra,v1
2,Algebra & Equations,Quadratic equations,Solve quadratics by formula,Algebra,v1
3,Algebra & Equations,Inequalities,Solve and represent inequalities,Algebra,v1
4,Algebra & Equations,Algebraic manipulation,Simplify/factorise/expand expressions,Algebra,v1
5,Algebra & Equations,Exponents & surds,Simplify exponential and surd expressions,Algebra,v1
6,Number Patterns & Sequences,Arithmetic sequences,Find nth term / sum of arithmetic sequence,Patterns,v1
7,Number Patterns & Sequences,Geometric sequences,Find nth term / sum of geometric sequence,Patterns,v1
8,Number Patterns & Sequences,Quadratic patterns,Determine quadratic pattern rules,Patterns,v1
9,Functions & Graphs,Linear functions,Interpret or sketch linear functions,Functions,v1


In [3]:
assert taxonomy_df["topic"].isna().sum() == 0
assert taxonomy_df["subtopic"].isna().sum() == 0
assert taxonomy_df.duplicated(["topic", "subtopic", "skill"]).sum() == 0

print("Taxonomy validation passed")
print(taxonomy_df["topic"].value_counts())

Taxonomy validation passed
topic
Algebra & Equations            6
Functions & Graphs             5
Calculus                       4
Trigonometry                   4
Number Patterns & Sequences    3
Finance                        3
Probability                    3
Euclidean Geometry             3
Analytical Geometry            3
Statistics                     2
Mixed / Multi-topic            1
Unmapped                       1
Name: count, dtype: int64


In [4]:
questions_df = pd.read_csv(question_structure_path)

# Stable question_id
def make_question_id(row):
    qn = row.get("question_number")
    subq = row.get("subquestion")
    qn_part = f"Q{int(qn)}" if pd.notna(qn) else "QNA"
    sub_part = str(subq) if pd.notna(subq) else "main"
    return f"{row['document_id']}__{qn_part}__{sub_part}"

questions_df["question_id"] = questions_df.apply(make_question_id, axis=1)

print("Question records:", len(questions_df))
print("Unique question_id:", questions_df["question_id"].nunique())
display(questions_df.head(3))

Question records: 417
Unique question_id: 414


,document_id,year,paper,document_type,source_text_file,extraction_quality,question_number,subquestion,marks,marks_all_found,question_text,question_char_count,segment_index,segmentation_status,question_id
0,2023_nov_p1_exam_maths,2023,P1,exam,data\processed\ocr\text\2023_nov_p1_exam_maths...,ocr_good,1.0,1.1,3.0,[3],1.1 Solve for x:\n\n11.1 x +x-12=0 (3),36.0,1.0,ok,2023_nov_p1_exam_maths__Q1__1.1
1,2023_nov_p1_exam_maths,2023,P1,exam,data\processed\ocr\text\2023_nov_p1_exam_maths...,ocr_good,1.0,1.1.2,4.0,[4],1.1.2 3x*-2x=6 (answers correct to TWO decimal...,58.0,2.0,ok,2023_nov_p1_exam_maths__Q1__1.1.2
2,2023_nov_p1_exam_maths,2023,P1,exam,data\processed\ocr\text\2023_nov_p1_exam_maths...,ocr_good,1.0,1.1.3,4.0,[4],1.1.3 V2xt+1 =x-1 (4),21.0,3.0,ok,2023_nov_p1_exam_maths__Q1__1.1.3


In [5]:
gold_source = questions_df[
    (questions_df["document_id"] == "2025_nov_p1_exam_maths")
    & (questions_df["segmentation_status"] == "ok")
].copy()

print("2025 P1 candidate rows:", len(gold_source))
display(
    gold_source[
        ["question_id", "question_number", "subquestion", "marks", "question_text"]
    ].head(15)
)

2025 P1 candidate rows: 50


,question_id,question_number,subquestion,marks,question_text
260,2025_nov_p1_exam_maths__Q1__1.1.2,1.0,1.1.2,NaN,1.1.2 5x? +2=-9x (correct to TWO decimal places)
261,2025_nov_p1_exam_maths__Q1__1.1.3,1.0,1.1.3,NaN,1.1.3 8x? > 2x
262,2025_nov_p1_exam_maths__Q1__1.1.4,1.0,1.1.4,NaN,1.1.4 2.27* -9.2* +4=0\n\n[ft 1
263,2025_nov_p1_exam_maths__Q1__1.1.5,1.0,1.1.5,NaN,1.1.5 —+2=—=\nx Vx
264,2025_nov_p1_exam_maths__Q1__1.2,1.0,1.2,NaN,1.2 Calculate the values of x and y if:\n\ne x...
265,2025_nov_p1_exam_maths__Q2__2.1,2.0,2.1,NaN,2.1 Given the infinite geometric series: (f+ 1...
266,2025_nov_p1_exam_maths__Q2__2.1.1,2.0,2.1.1,NaN,2.1.1 Show that t=—2
267,2025_nov_p1_exam_maths__Q2__2.1.2,2.0,2.1.2,NaN,"2.1.2 Calculate the value of 7,,. Write your a..."
268,2025_nov_p1_exam_maths__Q2__2.1.3,2.0,2.1.3,NaN,2.1.3 Calculate the sum of the infinite series...
269,2025_nov_p1_exam_maths__Q2__2.2,2.0,2.2,NaN,2.2 Given }'(4p-1)= 26 675\npak


In [6]:
gold_template = gold_source[
    [
        "question_id",
        "document_id",
        "year",
        "paper",
        "question_number",
        "subquestion",
        "marks",
        "question_text",
    ]
].copy()

# Manual label columns
gold_template["topic_gold"] = ""
gold_template["subtopic_gold"] = ""
gold_template["skill_gold"] = ""
gold_template["secondary_topic_gold"] = ""
gold_template["command_verb_gold"] = ""
gold_template["structure_type_gold"] = ""
gold_template["notes"] = ""
gold_template["labeler"] = ""
gold_template["labelled_at"] = ""

gold_template.to_csv(gold_template_path, index=False)
print("Gold template saved:", gold_template_path)
print("Rows to label:", len(gold_template))

Gold template saved: c:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\data\processed\gold_sets\gold_2025_p1_template.csv
Rows to label: 50


## Gold-set labelling instructions

Manually label: `data/processed/gold_sets/gold_2025_p1_template.csv`

For each row fill:
- `topic_gold` from taxonomy topics
- `subtopic_gold` from taxonomy subtopics
- `skill_gold` short assessed skill
- `secondary_topic_gold` only if needed
- `command_verb_gold` (calculate/determine/sketch/show/prove/...)
- `structure_type_gold` (routine_calculation/multi_step/interpretation/proof/...)

Rules:
1. Label the **assessed** skill, not every prerequisite
2. Use taxonomy topics only
3. If unsure, use `Unmapped` and explain in notes
4. Save completed file as:
   `data/processed/gold_sets/gold_2025_p1_labelled.csv`

In [7]:
if gold_labelled_path.exists():
    gold_df = pd.read_csv(gold_labelled_path)
    print("Labelled gold set loaded:", len(gold_df))
    print("Topic coverage:")
    print(gold_df["topic_gold"].value_counts(dropna=False))
else:
    print("Waiting for manual labels at:")
    print(gold_labelled_path)

Waiting for manual labels at:
c:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\data\processed\gold_sets\gold_2025_p1_labelled.csv


In [8]:
from pathlib import Path
import re
import json
from datetime import datetime, timezone

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
QUESTIONS_DIR = PROCESSED_DIR / "questions"
TAXONOMY_DIR = PROCESSED_DIR / "taxonomy"
GOLD_DIR = PROCESSED_DIR / "gold_sets"
MAPPED_DIR = PROCESSED_DIR / "mapped"
MAPPED_DIR.mkdir(parents=True, exist_ok=True)

gold_path = GOLD_DIR / "gold_2025_p1_labelled.csv"
question_structure_path = QUESTIONS_DIR / "question_structure.csv"
taxonomy_path = TAXONOMY_DIR / "taxonomy_math_v1.csv"

gold_df = pd.read_csv(gold_path)
questions_df = pd.read_csv(question_structure_path)

print("Gold rows:", len(gold_df))
print("Question structure rows:", len(questions_df))
print("\nGold topic distribution:")
print(gold_df["topic_gold"].value_counts())

assert len(gold_df) > 0, "Gold set is empty"
assert gold_df["topic_gold"].isna().sum() == 0, "Missing topic_gold labels"
print("\nGold set validation passed")

Gold rows: 49
Question structure rows: 417

Gold topic distribution:
topic_gold
Functions & Graphs             15
Calculus                       10
Number Patterns & Sequences     9
Algebra & Equations             6
Probability                     5
Finance                         4
Name: count, dtype: int64

Gold set validation passed


In [9]:
taxonomy_rows = [
    ("Algebra & Equations", "Linear equations", "Solve linear equations", "Algebra", "v1.1"),
    ("Algebra & Equations", "Quadratic equations", "Solve quadratic equations", "Algebra", "v1.1"),
    ("Algebra & Equations", "Inequalities", "Solve inequalities", "Algebra", "v1.1"),
    ("Algebra & Equations", "Exponents & surds", "Simplify/solve exponential and surd expressions", "Algebra", "v1.1"),
    ("Algebra & Equations", "Algebraic manipulation", "Simplify/factorise/expand expressions", "Algebra", "v1.1"),
    ("Algebra & Equations", "Simultaneous equations", "Solve simultaneous equations", "Algebra", "v1.1"),

    ("Number Patterns & Sequences", "Arithmetic sequences", "Work with arithmetic sequences/series", "Patterns", "v1.1"),
    ("Number Patterns & Sequences", "Geometric sequences", "Work with geometric sequences/series", "Patterns", "v1.1"),
    ("Number Patterns & Sequences", "Quadratic sequences", "Work with quadratic sequences", "Patterns", "v1.1"),
    ("Number Patterns & Sequences", "Series & sigma notation", "Evaluate series and sigma notation", "Patterns", "v1.1"),

    ("Functions & Graphs", "Linear functions", "Interpret/sketch linear functions", "Functions", "v1.1"),
    ("Functions & Graphs", "Quadratic functions", "Interpret/sketch parabolas", "Functions", "v1.1"),
    ("Functions & Graphs", "Hyperbola", "Interpret/sketch hyperbolas", "Functions", "v1.1"),
    ("Functions & Graphs", "Exponential functions", "Interpret/sketch exponential functions", "Functions", "v1.1"),
    ("Functions & Graphs", "Logarithmic functions", "Interpret/sketch logarithmic functions", "Functions", "v1.1"),
    ("Functions & Graphs", "Inverse functions", "Determine/interpret inverse functions", "Functions", "v1.1"),
    ("Functions & Graphs", "Transformations", "Apply function transformations", "Functions", "v1.1"),
    ("Functions & Graphs", "Graph interpretation", "Interpret graphs and intercepts/inequalities", "Functions", "v1.1"),

    ("Finance", "Compound interest", "Apply compound growth/decay", "Finance", "v1.1"),
    ("Finance", "Annuities", "Present/future value of annuities", "Finance", "v1.1"),
    ("Finance", "Present/future value", "Loan and investment timing calculations", "Finance", "v1.1"),

    ("Calculus", "First principles", "Differentiate from first principles", "Calculus", "v1.1"),
    ("Calculus", "Rules of differentiation", "Apply differentiation rules", "Calculus", "v1.1"),
    ("Calculus", "Cubic graphs", "Analyse cubic functions with calculus", "Calculus", "v1.1"),
    ("Calculus", "Tangents", "Determine tangent equations", "Calculus", "v1.1"),
    ("Calculus", "Optimisation", "Solve optimisation problems", "Calculus", "v1.1"),

    ("Probability", "Basic probability", "Compute simple probabilities", "Probability", "v1.1"),
    ("Probability", "Contingency tables", "Use contingency tables", "Probability", "v1.1"),
    ("Probability", "Tree diagrams", "Use tree diagrams", "Probability", "v1.1"),
    ("Probability", "Counting principles", "Use counting principles", "Probability", "v1.1"),

    ("Trigonometry", "Identities", "Prove/apply identities", "Trigonometry", "v1.1"),
    ("Trigonometry", "Equations", "Solve trig equations", "Trigonometry", "v1.1"),
    ("Euclidean Geometry", "Circle geometry", "Apply circle theorems", "Geometry", "v1.1"),
    ("Analytical Geometry", "Distance / midpoint / gradient", "Apply analytic geometry formulae", "Analytical Geometry", "v1.1"),
    ("Statistics", "Regression / correlation", "Interpret regression/correlation", "Statistics", "v1.1"),

    ("Mixed / Multi-topic", "Mixed", "Multi-topic assessed skill", "Mixed", "v1.1"),
    ("Unmapped", "Unmapped", "Insufficient evidence", "Unmapped", "v1.1"),
]

taxonomy_df = pd.DataFrame(
    taxonomy_rows,
    columns=["topic", "subtopic", "skill", "caps_section", "taxonomy_version"]
)
taxonomy_df.to_csv(taxonomy_path, index=False)
print("Taxonomy v1.1 saved:", len(taxonomy_df), "rows")

Taxonomy v1.1 saved: 37 rows


In [10]:
def map_topic(text: str):
    """
    Rule-based topic mapper.
    Returns: topic, subtopic, method, rule_id, confidence
    """
    t = (text or "").lower()

    rules = [
        # Calculus
        (r"first principles|f'\(x\).*first|from first principles", "Calculus", "First principles", "symbolic_calc_fp", "high"),
        (r"\bf'\(|dy/dx|derivative|differentiate|concave|turning point|optimisation|maximize|maximise|tangent to", "Calculus", "Rules of differentiation", "keyword_calc", "high"),

        # Probability
        (r"contingency|independent events|mutually exclusive", "Probability", "Contingency tables", "keyword_prob_table", "high"),
        (r"tree diagram", "Probability", "Tree diagrams", "keyword_prob_tree", "high"),
        (r"arrangements|counting principle|number of ways|permutation|combination", "Probability", "Counting principles", "keyword_prob_count", "high"),
        (r"probability that|p\(", "Probability", "Basic probability", "keyword_prob", "medium"),

        # Finance
        (r"compound interest|accumulated amount|future value|present value|annuity|loan|repay", "Finance", "Compound interest", "keyword_finance", "high"),

        # Sequences
        (r"quadratic sequence|second difference", "Number Patterns & Sequences", "Quadratic sequences", "keyword_quad_seq", "high"),
        (r"geometric sequence|infinite series|common ratio", "Number Patterns & Sequences", "Geometric sequences", "keyword_geo_seq", "high"),
        (r"arithmetic sequence|common difference|t_?\d", "Number Patterns & Sequences", "Arithmetic sequences", "keyword_arith_seq", "medium"),
        (r"sigma|sum of the series|s_?\d", "Number Patterns & Sequences", "Series & sigma notation", "keyword_series", "medium"),

        # Functions
        (r"log\s*\(|\blog\b|logarithmic", "Functions & Graphs", "Logarithmic functions", "keyword_log", "high"),
        (r"inverse of|f\s*[⁻\^-]?1|f\^\{\s*-1\s*\}", "Functions & Graphs", "Inverse functions", "keyword_inverse", "high"),
        (r"hyperbola|asymptote", "Functions & Graphs", "Hyperbola", "keyword_hyperbola", "medium"),
        (r"parabola|quadratic function|axis of symmetry", "Functions & Graphs", "Quadratic functions", "keyword_parabola", "medium"),
        (r"translated|translation|transformation", "Functions & Graphs", "Transformations", "keyword_transform", "medium"),
        (r"domain|range|intercept|sketch the graph|draw the graph", "Functions & Graphs", "Graph interpretation", "keyword_graph", "medium"),

        # Algebra
        (r"simultaneous|solve for x and y", "Algebra & Equations", "Simultaneous equations", "keyword_simultaneous", "high"),
        (r"inequalit|>|<|≥|≤", "Algebra & Equations", "Inequalities", "keyword_inequality", "medium"),
        (r"surd|√|square root", "Algebra & Equations", "Exponents & surds", "keyword_surd", "medium"),
        (r"quadratic formula|factoris|factoriz|\(.*\)\(.*\)\s*=\s*0|solve for x", "Algebra & Equations", "Quadratic equations", "keyword_quadratic", "medium"),
    ]

    for pattern, topic, subtopic, rule_id, conf in rules:
        if re.search(pattern, t, flags=re.IGNORECASE):
            return topic, subtopic, "rule", rule_id, conf

    return "Unmapped", "Unmapped", "rule", "no_match", "low"


def extract_command_verb(text: str):
    verbs = [
        "calculate", "determine", "solve", "show", "prove", "sketch", "draw",
        "write down", "describe", "explain", "hence", "deduce", "evaluate",
        "simplify", "factorise", "factorize", "expand"
    ]
    t = (text or "").lower()
    for v in verbs:
        if re.search(rf"\b{re.escape(v)}\b", t):
            return v
    return None

In [11]:
pred_rows = []
for _, row in gold_df.iterrows():
    topic, subtopic, method, rule_id, conf = map_topic(row["question_text"])
    pred_rows.append({
        "question_id": row["question_id"],
        "topic_gold": row["topic_gold"],
        "topic_pred": topic,
        "subtopic_gold": row.get("subtopic_gold"),
        "subtopic_pred": subtopic,
        "rule_id": rule_id,
        "confidence": conf,
        "match_topic": row["topic_gold"] == topic,
    })

pred_df = pd.DataFrame(pred_rows)
accuracy = pred_df["match_topic"].mean()

print(f"Topic accuracy vs gold: {accuracy:.1%}")
print("\nConfusion-style counts:")
print(pd.crosstab(pred_df["topic_gold"], pred_df["topic_pred"]))

print("\nMisses:")
display(pred_df[~pred_df["match_topic"]][
    ["question_id", "topic_gold", "topic_pred", "rule_id", "confidence"]
])

Topic accuracy vs gold: 51.0%

Confusion-style counts:
topic_pred                   Algebra & Equations  Calculus  Finance  \
topic_gold                                                            
Algebra & Equations                            5         0        0   
Calculus                                       1         5        0   
Finance                                        0         0        2   
Functions & Graphs                             2         0        0   
Number Patterns & Sequences                    0         0        0   
Probability                                    0         0        0   

topic_pred                   Functions & Graphs  Number Patterns & Sequences  \
topic_gold                                                                     
Algebra & Equations                           0                            0   
Calculus                                      0                            0   
Finance                                       0         

,question_id,topic_gold,topic_pred,rule_id,confidence
5,2025_P1_Q1.2,Algebra & Equations,Unmapped,no_match,low
6,2025_P1_Q2.1.1,Number Patterns & Sequences,Unmapped,no_match,low
7,2025_P1_Q2.1.2,Number Patterns & Sequences,Unmapped,no_match,low
9,2025_P1_Q2.2.1,Number Patterns & Sequences,Unmapped,no_match,low
10,2025_P1_Q2.2.2,Number Patterns & Sequences,Unmapped,no_match,low
11,2025_P1_Q3.1,Number Patterns & Sequences,Unmapped,no_match,low
12,2025_P1_Q3.2,Number Patterns & Sequences,Unmapped,no_match,low
13,2025_P1_Q3.3,Number Patterns & Sequences,Unmapped,no_match,low
14,2025_P1_Q3.4,Number Patterns & Sequences,Unmapped,no_match,low
15,2025_P1_Q4.1,Functions & Graphs,Unmapped,no_match,low


In [12]:
TARGET = 0.80
print(f"Target macro/topic agreement: {TARGET:.0%}")
print(f"Observed topic accuracy: {accuracy:.1%}")

if accuracy >= TARGET:
    print("PASS — proceed to full mapping")
else:
    print("BELOW TARGET — refine rules before Notebook 05")

Target macro/topic agreement: 80%
Observed topic accuracy: 51.0%
BELOW TARGET — refine rules before Notebook 05
